# dfs-three-set-toposort — worked example 3: Topological sort over a DAG with two independent roots sharing a common descendant

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dfs-three-set-toposort`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a graph has multiple entry points (multiple roots), calling `topological_sort` from any single node only reachable portion. If you want a complete ordering, you call `visit` on each root in turn; the `perm` set ensures a shared descendant is processed only once regardless of how many paths lead to it.

## Worked solution

Graph: R1 → X, R2 → X, X → L (leaf). We want a combined ordering that includes both roots.

**Call `visit(R1)`**: R1 → X → L. L appended, X appended, R1 appended. Result: `[L, X, R1]`.

**Call `visit(R2)`**: R2 is fresh. Its child X is already in `perm` — `visit(X)` returns immediately. R2 appended. Result: `[L, X, R1, R2]`.

**Why this is correct.** X appears exactly once even though both R1 and R2 would independently lead to it. The `perm` short-circuit handles the diamond and the shared-descendant cases identically. The leaf L is first, both roots come after X.

In [ ]:
def topological_sort_multi_root(roots, children_map):
    result = []
    perm = set()
    temp = set()

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError(f'Cycle at {node!r}')
        temp.add(nid)
        for child in children_map.get(node, []):
            visit(child)
        temp.discard(nid)
        perm.add(nid)
        result.append(node)

    for root in roots:
        visit(root)
    return result

# Graph: R1->X, R2->X, X->L
nodes = {'R1': 'R1', 'R2': 'R2', 'X': 'X', 'L': 'L'}
children = {'R1': ['X'], 'R2': ['X'], 'X': ['L'], 'L': []}

# Use actual string objects as keys (id-based lookup works on interned strings in CPython, but
# to be safe we use a string-keyed children_map and wrap nodes in unique objects).
class Node:
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name

R1, R2, X, L = Node('R1'), Node('R2'), Node('X'), Node('L')
children_obj = {R1: [X], R2: [X], X: [L], L: []}

ordering = topological_sort_multi_root([R1, R2], children_obj)
print('Multi-root order:', ordering)
print('L is first:', ordering[0] is L)
print('X before both roots:', ordering.index(X) < ordering.index(R1) and ordering.index(X) < ordering.index(R2))
print('All four nodes present:', set(ordering) == {R1, R2, X, L})